# Feature Engineering

Feature engineering is the process of using domain knowledge to create, transform, or combine raw features into new ones that make it easier for a model to learn patterns.

**Rule of thumb**: Spend 80% of your time on data and features, 20% on model tuning. A simple model with great features almost always beats a complex model with raw features!

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import skew
import os, sys
print(sys.executable)

# Load the cleaned but un-engineered data
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
Y_train = pd.read_csv('../data/processed/Y_train.csv').squeeze()

n_train = len(X_train)

# Combine for consistent engineering
all_data = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)

print(f"Combined shape before FE : {all_data.shape}")

c:\Users\Chyi\anaconda3\envs\kaggle\python.exe
Combined shape before FE : (2919, 79)


In [2]:
# ── Total square footage ──────────────────────────────────────
all_data['TotalSF']    = (all_data['TotalBsmtSF'] +
                          all_data['1stFlrSF'] +
                          all_data['2ndFlrSF'])

# ── Total porch area ─────────────────────────────────────────
all_data['TotalPorch'] = (all_data['OpenPorchSF'] +
                          all_data['EnclosedPorch'] +
                          all_data['3SsnPorch'] +
                          all_data['ScreenPorch'] +
                          all_data['WoodDeckSF'])

# ── Total bathrooms ──────────────────────────────────────────
all_data['TotalBath']  = (all_data['FullBath'] +
                          all_data['BsmtFullBath'] +
                          0.5 * all_data['HalfBath'] +
                          0.5 * all_data['BsmtHalfBath'])

print("✅ Area features done")

✅ Area features done


In [3]:
all_data['HouseAge']     = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge']     = all_data['YrSold'] - all_data['YearRemodAdd']
all_data['WasRemodeled'] = (all_data['YearRemodAdd'] != all_data['YearBuilt']).astype(int)
all_data['IsNew']        = (all_data['YearBuilt']    == all_data['YrSold']).astype(int)

print("✅ Time features done")

✅ Time features done


In [4]:
all_data['QualxArea']    = all_data['OverallQual'] * all_data['GrLivArea']
all_data['QualxTotalSF'] = all_data['OverallQual'] * all_data['TotalSF']
all_data['GarageScore']  = all_data['GarageQual']  * all_data['GarageArea']
all_data['BsmtScore']    = all_data['BsmtQual']    * all_data['TotalBsmtSF']

print("✅ Interaction features done")

✅ Interaction features done


In [5]:
all_data['HasPool']      = (all_data['PoolArea']   > 0).astype(int)
all_data['HasGarage']    = (all_data['GarageArea'] > 0).astype(int)
all_data['HasFireplace'] = (all_data['Fireplaces']  > 0).astype(int)
all_data['Has2ndFloor']  = (all_data['2ndFlrSF']   > 0).astype(int)
all_data['HasPorch']     = (all_data['TotalPorch']  > 0).astype(int)

print("✅ Binary flags done")

✅ Binary flags done


In [6]:
# Avoid division by zero with + 1
all_data['LivAreaRatio'] = all_data['GrLivArea']   / (all_data['TotalSF']      + 1)
all_data['BsmtFinRatio'] = all_data['BsmtFinSF1']  / (all_data['TotalBsmtSF']  + 1)
all_data['GarageRatio']  = all_data['GarageArea']  / (all_data['TotalSF']      + 1)

print("✅ Ratio features done")

✅ Ratio features done


In [7]:
num_cols    = all_data.select_dtypes(include=np.number).columns
skewness    = all_data[num_cols].apply(skew).sort_values(ascending=False)
high_skew   = skewness[skewness.abs() > 0.75].index

print(f"Applying log1p to {len(high_skew)} skewed features...")

for col in high_skew:
    all_data[col] = np.log1p(all_data[col].clip(lower=0))  # clip negatives first

print("✅ Skew correction done")

Applying log1p to 63 skewed features...
✅ Skew correction done


In [8]:
print(f"\nShape before FE : {n_train, X_train.shape[1]}")
print(f"Shape after FE  : {all_data.shape}")
print(f"New features    : {all_data.shape[1] - X_train.shape[1]}")
print(f"\nNew columns added:")
new_cols = [c for c in all_data.columns if c not in X_train.columns]
for c in new_cols:
    print(f"  + {c}")


Shape before FE : (1460, 79)
Shape after FE  : (2919, 98)
New features    : 19

New columns added:
  + TotalSF
  + TotalPorch
  + TotalBath
  + HouseAge
  + RemodAge
  + WasRemodeled
  + IsNew
  + QualxArea
  + QualxTotalSF
  + GarageScore
  + BsmtScore
  + HasPool
  + HasGarage
  + HasFireplace
  + Has2ndFloor
  + HasPorch
  + LivAreaRatio
  + BsmtFinRatio
  + GarageRatio


In [9]:
X_train_fe = all_data.iloc[:n_train].reset_index(drop=True)
X_test_fe  = all_data.iloc[n_train:].reset_index(drop=True)

os.makedirs('../data/processed/feature_engineering', exist_ok=True)

X_train_fe.to_csv('../data/processed/feature_engineering/X_train_fe.csv', index=False)
X_test_fe.to_csv('../data/processed/feature_engineering/X_test_fe.csv',   index=False)

print(f"✅ Saved!")
print(f"   X_train_fe : {X_train_fe.shape}")
print(f"   X_test_fe  : {X_test_fe.shape}")

✅ Saved!
   X_train_fe : (1460, 98)
   X_test_fe  : (1459, 98)
